# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K-0-A/My-Flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why
**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The search question I want to answer: **which pages should a human review first** for refresh, expansion, protection, pruning, or monitoring?

I am picking this lane (not freestyle) because it matches the decision FlyRank already has to make every week, and because Week 1 already showed the gap: a hand-written rule and a learned ranker disagree a lot on the top 50. The unit of analysis is **one content item (page)** in a trailing window, grouped by pseudonymized `client_id` so a client's pages are not split across train and test. The output is a **ranked review queue** with a suggested action and a reason I can read — not a black-box "decline probability" with no next step. I can confirm or change this through Week 4; for now this is the lane that makes the next seven weeks about a real queue, not about training a model for its own sake.

In [1]:
# Lane check (not the framing paragraph — that lives in the markdown cell above).
LANE = "Lane 2: Refresh / Content Opportunity Scoring"
UNIT = "one row = one pseudonymized content item (page)"
OUTPUT = "ranked review queue + suggested action + readable reason"
print(LANE)
print("Unit of analysis:", UNIT)
print("Intended output:", OUTPUT)
print("Provisional until end of Week 4: yes")


Lane 2: Refresh / Content Opportunity Scoring
Unit of analysis: one row = one pseudonymized content item (page)
Intended output: ranked review queue + suggested action + readable reason
Provisional until end of Week 4: yes


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** given thousands of pages, which ones should go to a human **first** this week?

**Who acts, and what they do:** a FlyRank editor or strategist (or the intern building the queue for them) opens the top of the list and chooses an action per page — refresh, expand, protect, prune, or just monitor. The model does not publish a rewrite; a person does.

**Cost of a wrong call:**
- **False alarm (high score, page is fine):** wasted editor hours on a page that did not need work, while a real declining page waits.
- **Miss (low score, page is slipping with real demand):** delayed refresh, more lost impressions/clicks, and a queue that looks "busy" while the valuable pages sit lower down.

Those two errors are not equal. Missing a visible, declining page usually costs more than reviewing one extra healthy page — so later I should care about **Precision@K** (are the first K pages actually worth a look?) and still watch misses among high-impression pages.

**Why data / ML can help at all:** a one-line rule ("oldest pages" or "highest search volume") is not enough. Week 1 already showed search-volume barely tracks actual impressions, and declining vs growing pages have almost the same median length. Many signals tangle together (position, volume, trend, content type, missingness). A learned score earns its place only if it **beats a transparent baseline queue** on a client-holdout split. If a simple rule wins, we keep the rule.

In [2]:
# Frame check: decision / actor / two error costs (prints only — prose is in the markdown above).
print("Decision: which pages a human reviews first")
print("Actor: editor/strategist taking refresh / expand / protect / prune / monitor")
print("Wrong-call costs: wasted hours (false alarm) vs delayed work on a slipping page (miss)")
print("Task type if we score a queue: ranking; metric Precision@K on a later observed outcome")
print("ML is optional: keep the hand-written baseline if it wins on holdout clients")

Decision: which pages a human reviews first
Actor: editor/strategist taking refresh / expand / protect / prune / monitor
Wrong-call costs: wasted hours (false alarm) vs delayed work on a slipping page (miss)
Task type if we score a queue: ranking; metric Precision@K on a later observed outcome
ML is optional: keep the hand-written baseline if it wins on holdout clients


## 3. Quick look at the data (2-3 real numbers)

Numbers below are computed live from `data/raw/content_refresh_anonymized.csv` (30,000 pages, 32 clients). They are why Lane 2 looks worth seven weeks:

1. **The queue is too big to review by hand.** About **54.2%** of pages are `trend_direction == "down"` (16,262 of 30,000). Even after a volume floor (`impressions_90d >= 100` so CTR/trend is not one-click noise), **13,152** pages are both declining and measurable. Nobody reviews 13k pages; a ranked top-K is the product.
2. **A naive "fix high-volume keywords" rule is a bad map.** Pearson correlation of `search_volume` vs `impressions_90d` is about **0.001**. Keyword demand estimates barely track the traffic a page actually got in this slice — so ranking by search volume would not put the right pages first.
3. **"Just write longer" is not the lever.** Median `word_count` is **2,909** for `down` vs **2,848** for `up`. Length looks almost the same, so a refresh scorer has to use other observed signals (position, impressions, freshness, type) — which is the messy case where a baseline-plus-model can help.

Week 1's pipeline comparison (same starter data, client-holdout): hand-written rule **Precision@50 ≈ 0.24** vs random forest **≈ 0.74**. That is a directional hint that a learned queue can beat a fixed rule here — not a promise it will still win on the warehouse.

In [11]:
import os, sys, subprocess
import pandas as pd

# ==========================================
# 1. COLAB SETUP & CLONING
# ==========================================
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    # Force Colab to start at /content before doing anything else
    os.chdir('/content')
    # Clone the repo if it hasn't been cloned yet
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    # Move into the repo folder
    os.chdir(REPO_DIR)
else:
    # Local logic: climb up until data/raw is found
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

# ==========================================
# 2. YOUR ANALYSIS SCRIPT
# ==========================================
csv_path = os.path.join("data", "raw", "content_refresh_anonymized.csv")
assert os.path.exists(csv_path), f"starter CSV not found from {os.getcwd()}"

df = pd.read_csv(csv_path)
n = len(df)
down = df["trend_direction"] == "down"
visible = df["impressions_90d"] >= 100
corr = df["search_volume"].corr(df["impressions_90d"])
wc = df.groupby("trend_direction")["word_count"].median()

print(f"Grain check: {n:,} rows, {df['client_id'].nunique()} clients, {df['content_id'].nunique():,} unique content_id")
print(f"1) Declining pages: {int(down.sum()):,} / {n:,} = {down.mean():.1%}")
print(f"   Declining AND impressions_90d >= 100: {int((down & visible).sum()):,}  (too many for a full manual review)")
print(f"2) Correlation(search_volume, impressions_90d) = {corr:.3f}  (near zero in this slice)")
print("3) Median word_count by trend_direction:")
print(wc.round(0).to_string())
print("   down vs up medians are close -> length is a weak lever here")
# IDs are pseudonyms — never print raw client_id / content_id values.

Grain check: 30,000 rows, 32 clients, 30,000 unique content_id
1) Declining pages: 16,262 / 30,000 = 54.2%
   Declining AND impressions_90d >= 100: 13,152  (too many for a full manual review)
2) Correlation(search_volume, impressions_90d) = 0.001  (near zero in this slice)
3) Median word_count by trend_direction:
trend_direction
down      2909.0
flat      2698.0
new       2239.0
stable    2912.0
up        2848.0
   down vs up medians are close -> length is a weak lever here


## 4. Careful words: what I can and can't claim

**This work can say (if the later notebooks hold up):**
- **Observed:** in this snapshot, these columns moved with decline / impressions / CTR in these grouped summaries.
- **Measured:** on a client-holdout split, queue A beat queue B on Precision@K (or did not).
- **Directional:** pages with this mix of signals were more often in the declining bucket *here* — useful for ordering a review list.
- **Decision-support:** "look at these K pages first, for this suggested action, because of these reasons." A human still decides.

**This work will not say:**
- We did not prove a Google ranking factor, and we will not claim we reverse-engineered the algorithm.
- A high score is **not** proof that a refresh will recover traffic. Decline is an observed outcome, not a guarantee that editing pays off.
- Product flags / health scores are **outputs to beat**, never features (that would be learning the existing rule).
- `trend_direction` and `trend_pct` are the **label source** for the starter declining flag — they must not be model features.
- `avg_position == 0` means missing position data, not rank zero. Rate columns are ×100 percentages (`ctr = 0.76` means 0.76%).
- No client names, URLs, or raw queries in anything public.

In [6]:
# Claim-discipline check (prints allowed language + a leakage reminder).
ALLOWED = ["observed", "measured", "directional", "decision-support"]
FORBIDDEN = ["proved Google's algorithm", "refresh will recover traffic", "health_score as a feature"]
print("Allowed claim words:", ", ".join(ALLOWED))
print("Will not claim:", "; ".join(FORBIDDEN))
print("Label trap: do not use trend_direction or trend_pct as model features")
print("Split: client_id is for grouped holdout only, not a feature")

Allowed claim words: observed, measured, directional, decision-support
Will not claim: proved Google's algorithm; refresh will recover traffic; health_score as a feature
Label trap: do not use trend_direction or trend_pct as model features
Split: client_id is for grouped holdout only, not a feature


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.